In [1]:
import os
from pathlib import Path
import jwst
print(jwst.__version__)
from jwst import datamodels
from jwst.datamodels import dqflags

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
# import natural units:
import natural_units as nu
# multi-core/thread:
import concurrent.futures
import re


1.15.1


In [2]:
# 全局变量，一波定义完
number_of_electron_bins = 30
# bin 的范围。用 range(dn_min, dn_max+1) to include dn_max
dn_min = -200
dn_max = 400

# define the mass and cross section grid we are working with.
log_m_min  = -3
log_m_max  = 1
n_m    = 17   # From 1e-3  to 10 GeV
#log cs shift from the balloon line
log_cs_min = -3
log_cs_max = 0
n_cs   = 25
m_grid  = np.logspace(log_m_min, log_m_max, n_m)
cs_grid = np.logspace(log_cs_min, log_cs_max, n_cs)
center_line = np.array([2.15504637e-23, 1.53030461e-23, 1.26359147e-23, 1.61669130e-23,
       2.40008514e-23, 4.03532201e-23, 6.95747264e-23, 1.40957345e-22,
       2.84518575e-22, 5.17381239e-22, 1.08351297e-21, 2.15203017e-21,
       4.12016417e-21, 7.78952222e-21, 1.45615810e-20, 2.70914228e-20,
       4.94000000e-20])

# fraction rescale for 0.1% and 0.05%
frac = '1e-4'
frac_rescale = float(frac) * 100 / 0.4   

In [3]:
#样本数量
sample_size = int(1e8)
# Generate pixel_value_raw by Poisson distribution:
def generate_raw_value(binned_signals):
    raw_value = np.zeros(sample_size)
    for i in range(len(binned_signals)):
        lambda_param = binned_signals[i]
        poisson_samples = np.random.poisson(lambda_param, sample_size)
        raw_value += (i+1) * poisson_samples
    return raw_value

binned_signal_path = Path('../data/binned_signal_w_shield_w_lindhard/')

def process_file(filename):
    file_stem = filename.stem
    pattern = r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2"
    # 使用 re.search() 匹配并提取
    match = re.search(pattern, file_stem)
    mDM_in_GeV = float(match.group(1))  # 第一个括号组匹配 mDM 的值
    sigma_e_in_cm2 = float(match.group(2))  # 第二个括号组匹配 sigma 的值
    m_index = np.argmin(np.abs(m_grid - mDM_in_GeV))
    cs_index = np.argmin(np.abs(cs_grid * center_line[m_index] - sigma_e_in_cm2))

    binned_signals = np.loadtxt(filename) * frac_rescale
    dm_sample = generate_raw_value(binned_signals)
    dm_poisson_counts, bin_edges = np.histogram(dm_sample, bins=range(dn_min, dn_max+1))
    dm_poisson_counts = dm_poisson_counts / sample_size   # to get PDF
    np.savetxt('./results_w_lin/DM_binned_shielding_frac_'+ frac +'/'+ str(m_index) + '_' + str(cs_index) +'.txt', dm_poisson_counts)
    print(file_stem[15:] + '  completed,     index' + str(m_index) + '_' + str(cs_index))

    return 0

binned_signal_files = [file for file in binned_signal_path.iterdir()]

In [ ]:
# 使用 ProcessPoolExecutor 并行处理文件，限制最大进程数为 10
with concurrent.futures.ProcessPoolExecutor(max_workers=16) as executor:
    # 获取文件夹中所有的文件路径
    file_paths = [file for file in binned_signal_path.iterdir() if file.is_file()]
    
    # 提交文件处理任务到进程池
    futures = {executor.submit(process_file, file): file for file in file_paths}
    
    # 逐个处理完成的任务
    for future in concurrent.futures.as_completed(futures):
        file = futures[future]
        try:
            result = future.result()  # 获取任务的返回值
            print(f"Finished processing {result}")
        except Exception as exc:
            print(f"Error processing {file}: {exc}")

binned_shielding_path = Path('./results/DM_binned_shielding/')
for filename in binned_shielding_path.iterdir():
    file_stem = filename.stem
    pattern = r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2"
    # 使用 re.search() 匹配并提取
    match = re.search(pattern, file_stem)
    mDM_in_GeV = float(match.group(1))  # 第一个括号组匹配 mDM 的值
    sigma_e_in_cm2 = float(match.group(2))  # 第二个括号组匹配 sigma 的值
    m_index = np.argmin(np.abs(m_grid - mDM_in_GeV))
    cs_index = np.argmin(np.abs(cs_grid * center_line[m_index] - sigma_e_in_cm2))
    new_file = filename.with_name(str(m_index) + '_' + str(cs_index) +'.txt')
    filename.rename(new_file)